# UnifyWeaver의 고급 재귀 패턴

이 노트북은 UnifyWeaver가 감지하고 최적화할 수 있는 4가지 주요 재귀 패턴을 보여줍니다:

1. **꼬리 재귀 (Tail Recursion)** - 누산기를 사용한 반복 루프
2. **선형 재귀 (Linear Recursion)** - 메모화가 적용된 단일 재귀 호출
3. **트리 재귀 (Tree Recursion)** - 구조의 여러 부분에 대한 다중 재귀 호출
4. **상호 재귀 (Mutual Recursion)** - 서술어들이 순환 구조로 서로를 호출

## 학습 목표

- 다양한 재귀 패턴 이해
- UnifyWeaver가 각 패턴을 감지하고 최적화하는 방법 확인
- 성능 특성 비교
- 각 패턴을 언제 사용해야 하는지 학습

## 설정

UnifyWeaver 환경을 초기화합니다.

In [ ]:
% Load initialization
['../init'].

% Load necessary modules
use_module(unifyweaver(core/recursive_compiler)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## 패턴 1: 꼬리 재귀 (Tail Recursion)

꼬리 재귀는 누산기(accumulator)를 사용하여 중간 결과를 전달하며, 재귀 호출이 함수의 **마지막 동작**입니다.

### 예제: 리스트의 항목 수 세기

In [ ]:
% Define tail-recursive count_items
:- dynamic count_items/3.

% Base case: empty list, return accumulator
count_items([], Acc, Acc).

% Recursive case: increment accumulator, recurse on tail
count_items([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count_items(T, Acc1, N).  % ← Tail position!

### Prolog에서 테스트

In [ ]:
% Test: count items in [a,b,c,d,e]
\+ \+ (
    count_items([a,b,c,d,e], 0, _N),
    format('Count: ~w~n', [_N])
).

### 패턴 감지 확인

In [ ]:
% Check if detected as tail recursive
\+ \+ (
    is_tail_recursive_accumulator(count_items/3, _AccInfo),
    format('Tail recursive: ~w~n', [_AccInfo])
).

### Bash로 컴파일

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(count_items/3, [], _BashCode),
    setup_call_cleanup(
        open('../output/count_items_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled count_items to Bash with tail recursion optimization')
).

### 생성된 Bash 테스트

In [ ]:
%%bash
source ../output/count_items_demo.sh
echo "Counting items in [a,b,c,d,e]:"
count_items "[a,b,c,d,e]" 0 ""

## 패턴 2: 선형 재귀 (Linear Recursion)

선형 재귀는 절당 **정확히 하나의** 재귀 호출을 가지며, 재귀 호출이 반환된 후에 추가 계산이 수행됩니다.

### 예제: 팩토리얼 (Factorial)

In [ ]:
% Define factorial
:- dynamic factorial/2.

% Base case
factorial(0, 1).

% Recursive case: exactly ONE recursive call
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),  % ← One recursive call
    F is N * F1.        % ← Computation after call

### Prolog에서 테스트

In [ ]:
% Test: factorial of 5
\+ \+ (
    factorial(5, _F),
    format('5! = ~w~n', [_F])
).

### 패턴 감지 확인

In [ ]:
% Check if detected as linear recursive
is_linear_recursive_streamable(factorial/2),
writeln('✓ Detected as linear recursion').

### Bash로 컴파일

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(factorial/2, [], _BashCode),
    % Keep function definitions only; Brush treats sourced scripts as direct execution
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Auto-execute when run directly (not when sourced)"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/factorial_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled factorial to Bash with fold-based linear recursion')
).

### 생성된 Bash 테스트

In [ ]:
%%bash
source ../output/factorial_demo.sh
echo "Factorial of 5:"
factorial 5 ""
echo ""
echo "Factorial of 10:"
factorial 10 ""

## 패턴 3: 트리 재귀 (Tree Recursion)

트리 재귀는 구조의 서로 다른 부분을 처리하기 위해 **여러 번의** 재귀 호출을 수행합니다.

### 예제: 트리 합계 (Tree Sum)

In [ ]:
% Define tree_sum for binary trees
% Tree format: [Value, LeftSubtree, RightSubtree] or []
:- dynamic tree_sum/2.

% Base case: empty tree has sum 0
tree_sum([], 0).

% Recursive case: sum = value + left_sum + right_sum
tree_sum([V, L, R], Sum) :-
    tree_sum(L, LS),   % ← First recursive call
    tree_sum(R, RS),   % ← Second recursive call
    Sum is V + LS + RS.

### Prolog에서 테스트

In [ ]:
% Test: tree_sum of [5, [3, [1, [], []], []], [2, [], []]]
%       5
%      / \
%     3   2
%    /
%   1
\+ \+ (
    tree_sum([5, [3, [1, [], []], []], [2, [], []]], _Sum),
    format('Tree sum: ~w (expected 11)~n', [_Sum])
).

### Bash로 컴파일

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(tree_sum/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/tree_sum_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled tree_sum to Bash with tree recursion')
).

### 생성된 Bash 테스트

In [ ]:
%%bash
source ../output/tree_sum_demo.sh
echo "Tree sum of [5,[3,[1,[],[]],[]],[2,[],[]]]:"
tree_sum "[5,[3,[1,[],[]],[]],[2,[],[]]]"

## 패턴 4: 상호 재귀 (Mutual Recursion)

상호 재귀는 둘 이상의 서술어가 순환하여 서로를 호출할 때 발생합니다.

### 예제: 짝수(Even)와 홀수(Odd)

In [ ]:
% Define mutually recursive is_even and is_odd
:- dynamic is_even/1.
:- dynamic is_odd/1.

% is_even base case
is_even(0).

% is_even recursive: N is even if N-1 is odd
is_even(N) :-
    N > 0,
    N1 is N - 1,
    is_odd(N1).  % ← Calls is_odd

% is_odd base case
is_odd(1).

% is_odd recursive: N is odd if N-1 is even
is_odd(N) :-
    N > 1,
    N1 is N - 1,
    is_even(N1).  % ← Calls is_even

### Prolog에서 테스트

In [ ]:
% Test even/odd
is_even(0), writeln('✓ 0 is even').
is_even(4), writeln('✓ 4 is even').
is_odd(3), writeln('✓ 3 is odd').
is_odd(7), writeln('✓ 7 is odd').

### 상호 재귀 확인

In [ ]:
% Build call graph and find SCCs
\+ \+ (
    use_module(unifyweaver(core/advanced/call_graph)),
    use_module(unifyweaver(core/advanced/scc_detection)),

    build_call_graph([is_even/1, is_odd/1], _Graph),
    format('Call graph: ~w~n', [_Graph]),

    find_sccs(_Graph, _SCCs),
    format('SCCs (mutual recursion groups): ~w~n', [_SCCs])
).

### Bash로 컴파일

In [ ]:
% Compile the mutual recursion group
\+ \+ (
    use_module(unifyweaver(core/advanced/mutual_recursion)),

    compile_mutual_recursion([is_even/1, is_odd/1], [], _BashCode),
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Main dispatch: route command line calls to functions"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/even_odd_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled is_even/is_odd to Bash with shared memoization')
).

### 생성된 Bash 테스트

In [ ]:
%%bash
source ../output/even_odd_demo.sh
echo "Testing is_even and is_odd:"
is_even 0 >/dev/null && echo "✓ 0 is even"
is_even 4 >/dev/null && echo "✓ 4 is even"
is_odd 3 >/dev/null && echo "✓ 3 is odd"
is_odd 7 >/dev/null && echo "✓ 7 is odd"
is_even 5 >/dev/null 2>&1 || echo "✓ 5 is not even"

## 패턴 비교

각 패턴의 특성을 비교해 보겠습니다:

| 패턴 | 재귀 호출 | 최적화 방식 | 공간 복잡도 | 적합한 용도 |
|:--------|:----------------|:-------------|:-----------------|:---------|
| **꼬리 재귀** | 1회 (꼬리 위치) | 반복 루프 | O(1) | 누산기, 선형 스캔 |
| **선형 재귀** | 1회 (임의 위치) | Fold + 메모화 | O(n) 메모 테이블 | 피보나치, 팩토리얼 |
| **트리 재귀** | 2회 이상 (구조 각 부분) | 구조적 분해 | O(depth) 스택 | 트리/그래프 연산 |
| **상호 재귀** | 1회 이상 (서술어 간) | 공유 메모화 | O(n) 공유 테이블 | 짝수/홀수, 상호 정의 |

## 패턴 감지 순서

UnifyWeaver는 다음 순서로 패턴 일치를 시도합니다:

1. **꼬리 재귀** (가장 효율적)
2. **선형 재귀** (금지되지 않은 경우)
3. **트리 재귀** (구조적)
4. **상호 재귀** (강결합 컴포넌트 SCC 감지)
5. **기본 재귀** (대체 폴백)

`forbid_linear_recursion/1`을 사용하여 감지 동작을 제어할 수 있습니다.

## 연습 과제: 직접 해보세요!

다음 서술어를 정의하고 컴파일해 보세요:

### 1. 꼬리 재귀 합계
```prolog
sum_list([], Acc, Acc).
sum_list([H|T], Acc, Sum) :-
    Acc1 is Acc + H,
    sum_list(T, Acc1, Sum).
```

### 2. 선형 재귀 피보나치
```prolog
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.
```

### 3. 트리 높이
```prolog
tree_height([], 0).
tree_height([_, L, R], H) :-
    tree_height(L, HL),
    tree_height(R, HR),
    H is max(HL, HR) + 1.
```

In [ ]:
% Your code here!


## 요약

이 노트북에서 배운 내용:

✅ UnifyWeaver의 4가지 주요 재귀 패턴

✅ Prolog에서 각 패턴을 정의하는 방법

✅ UnifyWeaver가 각 패턴을 감지하고 최적화하는 방법

✅ 각 패턴의 성능 특성

✅ 각 패턴을 언제 사용해야 하는지

## 다음 단계

고급 코드 분석 및 시각화에 대해 배우려면 **노트북 3: 호출 그래프 시각화**로 이동하세요!